In [2]:
#GRU_SED
import tensorflow as tf
from tensorflow.python.keras.layers import merge
from tensorflow.python.keras.layers.merge import Multiply
from tensorflow.python.keras.layers.core import *
from tensorflow.python.keras.models import *

import tensorflow.keras.backend as K
from tensorflow.python.keras.layers import Input,Dense,Reshape,Dropout, Embedding, LSTM, Bidirectional,Permute
from tensorflow.python.keras.layers import RepeatVector, TimeDistributed
from tensorflow.python.keras.layers.recurrent import GRU
from tensorflow.python.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam,SGD
import matplotlib.pyplot as plt
import pandas as pd
from pandas import concat, read_csv
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import sklearn.metrics as skm
import math
from math import sqrt
np.random.seed(6)
#1. load dataset
dataframe = read_csv('C:/Users/Administrator/Desktop/XI-2/data/PD-TVFEMD-BDX.csv', engine='python')
dataset = dataframe.values
dataset = dataset.astype('float32')
data = dataset.reshape(-1,6)


timestep = 3
dim = 6

#数据缩放 拆分输入X（7维）&输出Y（1维）
X = dataset[:,0:6]
scaler = MinMaxScaler(feature_range=(0, 1))
for i in range(dim):
    Xdata = X[:,i]
    Xdata = Xdata.reshape(-1,1)
    Xdata = scaler.fit_transform(Xdata)
    Xdata = Xdata.flatten()
    X[:,i] = Xdata
X_scaler = X.reshape(324,-1)
print(X_scaler.shape)

Y = dataset[:,5]
Y_scaler= (Y-np.min(Y))/(np.max(Y)-np.min(Y))
Y_scaler = Y_scaler.reshape(-1)
print(Y_scaler.shape)


#重构数据集
##timestep为时间步长
def create_X(seq, timestep):
    dataX = []
    for i in range(len(seq)-timestep):
        a = seq[i:(i+timestep)]
        # X按照顺序取值 每次在后面增加一个数据
        dataX.append(a)
    return np.array(dataX)

def create_Y(seq, timestep):
    dataY = []
    for i in range(len(seq)-timestep):
        # Y向后移动一位取值
        dataY.append(seq[i+timestep])
    return np.array(dataY)


# 将数据拆分成训练和测试，7/9作为训练数据
train_size = int(len(dataset) * 0.89)
test_size = len(dataset) - train_size
trainX, testX = X_scaler[0:train_size,:], X_scaler[train_size:len(dataset),:]
trainY, testY = Y_scaler[0:train_size], Y_scaler[train_size:len(dataset)]
print("原始训练集的长度：",train_size)
print("原始测试集的长度：",test_size)
#print(trainX,trainX.shape)
#print(trainY,trainY.shape)
#print(testX,testX.shape)
#print(testY,testY.shape)

train_X=create_X(trainX,timestep)#(246,6,6)
#print(train_X,train_X.shape)
train_Y=create_Y(trainY,timestep)#(246,)
#print(train_Y,train_Y.shape)
test_X=create_X(testX,timestep)#(66,6,6)
print(test_X,test_X.shape)
test_Y=create_Y(testY,timestep)#(66,)
print(test_Y,test_Y.shape)

(324, 6)
(324,)
原始训练集的长度： 288
原始测试集的长度： 36
[[[0.31573427 0.20798719 0.5296011  0.22493723 0.30868766 0.6665878 ]
  [0.35678926 0.27867207 0.4580318  0.22687894 0.33926573 0.6836537 ]
  [0.414593   0.36623043 0.55559206 0.71669203 0.41255692 0.6496603 ]]

 [[0.35678926 0.27867207 0.4580318  0.22687894 0.33926573 0.6836537 ]
  [0.414593   0.36623043 0.55559206 0.71669203 0.41255692 0.6496603 ]
  [0.3650726  0.40544453 0.62271154 0.99999994 0.4221195  0.44328737]]

 [[0.414593   0.36623043 0.55559206 0.71669203 0.41255692 0.6496603 ]
  [0.3650726  0.40544453 0.62271154 0.99999994 0.4221195  0.44328737]
  [0.31138846 0.34576085 0.4946666  0.55344605 0.35655078 0.3992847 ]]

 [[0.3650726  0.40544453 0.62271154 0.99999994 0.4221195  0.44328737]
  [0.31138846 0.34576085 0.4946666  0.55344605 0.35655078 0.3992847 ]
  [0.31991327 0.21769178 0.2777971  0.02371913 0.32418266 0.3963293 ]]

 [[0.31138846 0.34576085 0.4946666  0.55344605 0.35655078 0.3992847 ]
  [0.31991327 0.21769178 0.2777971  0.0